# UNSW-NB15 Pipeline — Per-Stage Colab Notebook

Each cell runs one pipeline stage. If a cell gets interrupted,
just re-run it — completed stages are skipped via `--resume`.

**Workflow:** Mount → Git pull → Install → Extract → Run stages 1-9 → Backup

Use `Runtime > Restart runtime` if you hit import issues after git pull.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Google Drive mounted.')

## Cell 2 — Clone / Pull from GitHub

In [ ]:
import os, subprocess

PROJECT_DIR = '/content/lstm_ids_project'
REPO_URL = 'https://github.com/Nico3783/lstm_ids_project.git'

try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
    AUTH_URL = REPO_URL.replace('https://', f'https://{GH_TOKEN}@') if GH_TOKEN else REPO_URL
except Exception:
    AUTH_URL = REPO_URL

if os.path.exists(PROJECT_DIR):
    print('Pulling latest...')
    r = subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
        subprocess.run(['git', '-C', PROJECT_DIR, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', AUTH_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3

## Cell 3 — Install Dependencies + Rename TF Shim

In [ ]:
import os

# Rename local TF shim so real TF is used
if os.path.isdir('tensorflow') and not os.path.isdir('tensorflow_shim_local'):
    os.rename('tensorflow', 'tensorflow_shim_local')
    print('Renamed tensorflow/ -> tensorflow_shim_local/')

!pip install -q -r requirements_colab.txt
print('Dependencies installed.')

## Cell 4 — Environment Validation

In [ ]:
import os, platform, sys

PROJECT = '/content/lstm_ids_project'
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['PYTHONPATH'] = PROJECT

# GPU
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f'GPU: {gpus[0].name}')
        tf.config.experimental.set_memory_growth(gpus[0], True)
    else:
        print('WARNING: No GPU.')
except Exception as e:
    print(f'TF check failed: {e}')

# RAM
try:
    import psutil
    print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB')
except ImportError:
    pass

# Disk
st = os.statvfs('.')
print(f'Disk free: {(st.f_bavail * st.f_frsize) / 1024**3:.1f} GB')

# Import check
from src.pipeline.checkpoint import CheckpointManager
print(f'Pipeline import OK — {len(CheckpointManager.STAGES)} stages')

## Cell 5 — Extract Raw Data (skip if exists)

In [ ]:
import os

DATA_DIR = 'data/raw/unsw_nb15'
if os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) >= 3:
    print('UNSW-NB15 data already exists — skipping extraction.')
else:
    import zipfile, glob
    # Try Drive backups
    candidates = [
        '/content/drive/MyDrive/lstm_raw.zip',
        '/content/drive/MyDrive/lstm_ids_project/data/raw/lstm_raw.zip',
        '/content/drive/MyDrive/unsw_nb15.zip',
    ]
    zip_path = next((p for p in candidates if os.path.exists(p)), None)
    if zip_path:
        print(f'Extracting {zip_path}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall('data/raw')
        print('Done.')
    else:
        print('WARNING: No UNSW-NB15 zip found on Drive.')
        print('Upload unsw_nb15.zip to /content/drive/MyDrive/ or to data/raw/unsw_nb15/')

# Verify
if os.path.isdir(DATA_DIR):
    files = os.listdir(DATA_DIR)
    print(f'UNSW-NB15 dir: {len(files)} files')
    for f in sorted(files):
        print(f'  {f}')
else:
    print('ERROR: data/raw/unsw_nb15/ missing. Upload data manually.')

## Cell 6 — Stage 1: Preprocessing

In [ ]:
import os, shutil
os.chdir('/content/lstm_ids_project')

# Delete old preprocessed data to force re-run with new code
out_dir = 'outputs/unsw_nb15'
preprocessed = os.path.join(out_dir, 'preprocessed')
checkpoints = os.path.join(out_dir, '.checkpoints')
models = os.path.join(out_dir, 'models')
for d in [preprocessed, checkpoints, models]:
    if os.path.isdir(d):
        shutil.rmtree(d)
        print(f'Deleted {d}')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage preprocessing --end-stage preprocessing --log-level INFO

## Cell 7 — Stage 2: Split 2D Data

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage split_save --end-stage split_save --log-level INFO

## Cell 8 — Stage 3: Scale + Build Sequences (per-split)

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage scale_sequences --end-stage scale_sequences --log-level INFO

## Cell 9 — Stage 5: Baselines (RF, SVM, LR)

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

# Keepalive thread (5 min heartbeat)
import threading, time
def _heartbeat():
    while True:
        print(f'[{time.strftime("%H:%M:%S")}] keepalive', flush=True)
        time.sleep(300)
threading.Thread(target=_heartbeat, daemon=True).start()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage baselines --end-stage baselines --log-level INFO

## Cell 10 — Stage 6: LSTM Training (longest stage ~20-40 min)

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

# Keepalive thread
import threading, time
def _heartbeat():
    while True:
        print(f'[{time.strftime("%H:%M:%S")}] keepalive', flush=True)
        time.sleep(300)
threading.Thread(target=_heartbeat, daemon=True).start()

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage lstm_train --end-stage lstm_train --epochs 100 --batch-size 64 --log-level INFO

# --- Incremental Drive backup after training ---
import shutil
DRIVE_BACKUP = '/content/drive/MyDrive/lstm_ids_results'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
for f in ['outputs/unsw_nb15/models/final/lstm_ids_model.keras',
          'outputs/unsw_nb15/models/checkpoints/best_model.keras',
          'outputs/unsw_nb15/models/checkpoints/periodic.keras',
          'outputs/unsw_nb15/models/final/model_metadata.json',
          'outputs/unsw_nb15/reports/logs/training_history.csv']:
    if os.path.exists(f):
        dst = os.path.join(DRIVE_BACKUP, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f'Backed up: {f} → {dst}')

## Cell 11 — Stage 7: Evaluation

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage evaluation --end-stage evaluation --log-level INFO

# --- Incremental Drive backup after evaluation ---
import shutil
DRIVE_BACKUP = '/content/drive/MyDrive/lstm_ids_results'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
eval_dir = 'outputs/unsw_nb15/tables'
if os.path.isdir(eval_dir):
    for f in os.listdir(eval_dir):
        src = os.path.join(eval_dir, f)
        dst = os.path.join(DRIVE_BACKUP, f)
        shutil.copy2(src, dst)
        print(f'Backed up: {src} → {dst}')

## Cell 12 — Stage 8: Visualization

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage visualization --end-stage visualization --log-level INFO

## Cell 13 — Stage 9: Export + Zip to Drive

In [ ]:
import os
os.chdir('/content/lstm_ids_project')

!PYTHONPATH=/content/lstm_ids_project python run_pipeline.py --dataset unsw_nb15 --resume --start-stage export --end-stage export --zip-drive --log-level INFO

## Cell 14 — Verify Drive Backup

In [ ]:
import os

zip_path = '/content/drive/MyDrive/lstm_ids_results/unsw_nb15.zip'
if os.path.exists(zip_path):
    size_gb = os.path.getsize(zip_path) / (1024**3)
    print(f'Backup OK: {zip_path} ({size_gb:.2f} GB)')
else:
    print(f'WARNING: {zip_path} not found.')

# Show local outputs
outputs_dir = 'outputs/unsw_nb15'
if os.path.isdir(outputs_dir):
    for root, dirs, files in os.walk(outputs_dir):
        for f in files:
            fp = os.path.join(root, f)
            sz = os.path.getsize(fp) / 1024
            print(f'  {fp} ({sz:.1f} KB)')